In [3]:
import pandas as pd
import numpy as np
from pulp import *

In [4]:
np.random.seed(42)
n = 50

df = pd.DataFrame({
    'supplier_id': [f'SUP_{i:03d}' for i in range(1, n+1)],
    'component_type': np.random.choice(
        ['Optical', 'Mechanical', 'Electronic', 'Pneumatic'], n),
    'lead_time_days': np.random.randint(14, 120, n),
    'lead_time_std_dev': np.random.randint(2, 20, n),
    'unit_cost_eur': np.random.randint(500, 50000, n),
    'on_time_delivery_rate': np.round(np.random.uniform(0.7, 1.0, n), 2),
    'defect_rate': np.round(np.random.uniform(0.01, 0.10, n), 3),
    'single_source': np.random.choice([True, False], n),
    'country': np.random.choice(
        ['Netherlands', 'Germany', 'Japan', 'USA', 'Taiwan'], n),
    'criticality': np.random.choice(['High', 'Medium', 'Low'], n)
})

In [5]:
# Create the problem
prob = LpProblem("Supplier_Selection", LpMinimize)

# Decision variables — 1 if supplier selected, 0 if not
suppliers = df['supplier_id'].tolist()
x = LpVariable.dicts("select", suppliers, cat='Binary')

# Objective: minimize weighted risk score
# (combination of lead time, defect rate, single source risk)
risk_scores = dict(zip(df['supplier_id'],
    df['lead_time_days'] * df['defect_rate']))

prob += lpSum([risk_scores[s] * x[s] for s in suppliers])

# Constraints
# 1. Budget constraint
costs = dict(zip(df['supplier_id'], df['unit_cost_eur']))
prob += lpSum([costs[s] * x[s] for s in suppliers]) <= 500000

# 2. Must cover all component types
for comp_type in df['component_type'].unique():
    relevant = df[df['component_type']==comp_type]['supplier_id'].tolist()
    prob += lpSum([x[s] for s in relevant]) >= 1

# Solve
prob.solve()

1

In [6]:
# Check solution status
print(f"Status: {LpStatus[prob.status]}")
print(f"Total Risk Score: {value(prob.objective):.2f}")
print(f"\n{'='*60}")
print("SELECTED SUPPLIERS:")
print(f"{'='*60}")

total_cost = 0
selected_count = 0

for s in suppliers:
    if x[s].value() == 1:
        row = df[df['supplier_id']==s].iloc[0]
        total_cost += row['unit_cost_eur']
        selected_count += 1
        print(f"\nSupplier: {s}")
        print(f"Component Type: {row['component_type']}")
        print(f"Lead Time: {row['lead_time_days']} days")
        print(f"Defect Rate: {row['defect_rate']*100:.1f}%")
        print(f"Unit Cost: €{row['unit_cost_eur']:,}")
        print(f"Criticality: {row['criticality']}")

print(f"\n{'='*60}")
print(f"Total Suppliers Selected: {selected_count}")
print(f"Total Cost: €{total_cost:,}")
print(f"Budget Remaining: €{500000 - total_cost:,}")

Status: Optimal
Total Risk Score: 2.24

SELECTED SUPPLIERS:

Supplier: SUP_006
Component Type: Pneumatic
Lead Time: 16 days
Defect Rate: 3.1%
Unit Cost: €22,018
Criticality: Low

Supplier: SUP_009
Component Type: Electronic
Lead Time: 20 days
Defect Rate: 5.6%
Unit Cost: €49,202
Criticality: High

Supplier: SUP_035
Component Type: Mechanical
Lead Time: 17 days
Defect Rate: 2.6%
Unit Cost: €43,441
Criticality: Medium

Supplier: SUP_036
Component Type: Optical
Lead Time: 15 days
Defect Rate: 1.2%
Unit Cost: €22,334
Criticality: High

Total Suppliers Selected: 4
Total Cost: €136,995
Budget Remaining: €363,005
